# 08 · Neurons and computation in superposition

**Paper:** "Superposition in a Privileged Basis" (pp. 31–34) and "Computation in
Superposition" (pp. 34–39).
**Experiments:** E07 `privileged_basis`, E08 `abs_computation`.

**Learning goals**

1. Understand why a ReLU on the hidden layer makes the neuron basis privileged, so W can be
   read neuron by neuron (p. 31).
2. Read neuron stack plots and watch neurons turn from monosemantic to polysemantic as
   features get sparser (pp. 32–33).
3. See the ReLU hidden layer model sidestep its ReLU when it can (p. 34).
4. Build |x| from ReLU neurons, and see what happens when there are more features than neurons
   can serve (pp. 35–38).

The two quick profiles take about 7 and 5 minutes on a laptop CPU.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition.experiments.runner import run_experiment
from superposition.runs import list_runs

pd.set_option("display.max_colwidth", 160)


REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    """Your latest complete quick-profile run of ``name`` with its figures, or a fresh one.

    Runs made without figures (``superposition run --no-figures``, or the replication tests)
    are skipped: this notebook shows the run's figures.
    """
    if REUSE:
        for run in reversed(list_runs(name)):
            quick = yaml.safe_load(run.config_path.read_text())["profile"] == "quick"
            if quick and any(run.figures_dir.glob("*.png")):
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["group", "id", "status", "value"]]

## 1. A privileged basis (E07, pp. 31–34)

Model: h = ReLU(Wx), x′ = ReLU(Wᵀh + b), with n = 10 features, m = 5 neurons and Iᵢ = 0.75ⁱ.
Rotating the hidden space no longer leaves the model unchanged, because the ReLU acts on each
neuron separately, so it is meaningful to ask which features a *neuron* carries.

**Predict:** at 1 − S = 1 (dense), how many features does each neuron carry? At 1 − S = 0.03?

In [ ]:
run7 = quick_run("privileged_basis")
criteria(run7)

In [ ]:
display(Image(filename=str(run7.figures_dir / "neurons.png")))
display(Image(filename=str(run7.figures_dir / "neuron_counts.png")))

In [ ]:
run7.read_table("summary")[
    [
        "feature_probability",
        "monosemantic",
        "polysemantic",
        "features_per_neuron",
        "represented",
        "loss",
    ]
]

Compare with the paper's rows on p. 33: all monosemantic when dense; then a few neurons start
to share features (the paper's "three features in two neurons" regime), until at the sparsest
settings every neuron is polysemantic.

**Sidestepping the ReLU (p. 34).** The model gains nothing from its hidden ReLU on this task
(it only reconstructs its input), so it avoids it when it can. The variant below gives the
hidden layer a bias c: the model makes c positive and keeps every neuron in its linear regime,
while the replication model (no hidden bias) cannot.

In [ ]:
doc = run7.read_acceptance()
sidestep = next(c for c in doc["criteria"] if c["id"] == "relu_sidestepping")
pd.DataFrame(sidestep["value"]).T

"clipped" is the share of (input, neuron) pairs where the ReLU zeroes a negative
pre-activation: with a positive hidden bias it is near zero, so the ReLU is effectively gone.

## 2. Computing |x| (E08 part a, p. 36)

Now the model must *compute* something nonlinear: y = |x| for inputs that are 0 or uniform on
[−1, 1], with h = ReLU(W₁x), y′ = ReLU(W₂h + b). Since |x| = ReLU(x) + ReLU(−x), each feature
needs two neurons: a positive side and a negative side.

**Predict:** with n = 3 features and m = 6 neurons, what do W₁ and W₂ look like after
footnote-15 normalization (each neuron's largest W₁ weight scaled to ±1)?

In [ ]:
run8 = quick_run("abs_computation")
criteria(run8)

In [ ]:
display(Image(filename=str(run8.figures_dir / "abs_n3_m6.png")))

In [ ]:
from superposition.experiments.abs_computation import TrainedAbs

small = TrainedAbs(**run8.load_checkpoint("n3_m6")).model()
x = torch.tensor([[[0.7, -0.3, 0.0]], [[-1.0, 0.5, 0.2]]])
with torch.no_grad():
    print("model:", small(x)[:, 0])
print("|x|  :", x.abs()[:, 0])

## 3. Computation in superposition (E08 part b, pp. 36–38)

n = 100 features (Iᵢ = 0.8ⁱ) and m = 40 neurons. Without superposition the model can compute
|x| for at most m/2 = 20 features. The p. 37 figure shows stack plots of |W₁| per neuron:
monosemantic when dense, polysemantic when sparse.

**Predict:** at 1 − S = 0.001, will the model compute |x| well for more than 20 features?

In [ ]:
display(Image(filename=str(run8.figures_dir / "abs_stack_plots.png")))
display(Image(filename=str(run8.figures_dir / "abs_capacity.png")))

In [ ]:
run8.read_table("metrics")

The neuron picture matches p. 37: dense models are monosemantic, sparse models' neurons carry
many features, often one large "primary" weight plus smaller "secondary" ones (p. 38). But
look at the capacity: E08 counts a feature as *computed well* when its loss is below half of
what predicting the mean would cost, and at the sparsest setting our models compute fewer than
m/2 features that well (see docs/REPRODUCTION.md for the exact numbers and status).

The diagnostic below explains much of it. At very sparse settings almost every input has one
active feature, so each **side** of each feature (x > 0 or x < 0) is trained only by its own
examples. With the output bias starting at 0 (SPEC §3), a side whose initial gain
Σₕ W₂[i,h] ReLU(±W₁[h,i]) is negative produces a negative pre-activation, the output ReLU
clips it, and no gradient ever reaches it.

In [ ]:
doc = run8.read_acceptance()
for key in ("b_side_diagnostics", "b_bias_init_variant"):
    row = next(c for c in doc["criteria"] if c["id"] == key)
    print(row["description"])
    display(pd.DataFrame(row["value"]).T)

## Exercises

**1.** In E07, the dense model has five monosemantic neurons for ten features. Which five
features get neurons, and why those?

<details><summary>Solution</summary>

The five most important (features 0–4): with Iᵢ = 0.75ⁱ each dedicated neuron should go to the
feature whose squared error costs the most. Check with
`run7.read_table("summary")["features_per_neuron"]` and the W panel of the dense row.
</details>

**2.** Why does a positive hidden bias make the ReLU hidden layer model "linear", and why is
the model better off that way on this task?

<details><summary>Solution</summary>

With c large enough, Wx + c > 0 for every input, so ReLU(Wx + c) = Wx + c: the hidden layer
is affine and the model becomes the ReLU output model x′ = ReLU(WᵀWx + Wᵀc + b), with an
effective bias Wᵀc + b. The ReLU output model can do everything the ReLU hidden model can and
more (negative hidden values are no longer lost), so gradient descent drives c up whenever it
is allowed to (p. 34).
</details>

**3.** Verify footnote 15 yourself: rescale neuron 0 of the n = 3 model by α = 3 (row of W₁
times α, column of W₂ divided by α) and check the outputs are unchanged.

<details><summary>Solution</summary>

```python
from superposition.models import AbsModel

m = small
x = torch.rand(8, 1, 3) * 2 - 1
before = m(x)
with torch.no_grad():
    m.W1[:, 0, :] *= 3.0
    m.W2[:, :, 0] /= 3.0
print(torch.allclose(m(x), before, atol=1e-6))
```

ReLU is positively homogeneous, ReLU(αz) = α ReLU(z) for α > 0, so the scale passes through
the neuron and is undone by W₂.
</details>

**4.** Using the side diagnostics, estimate how many feature sides would be computed at
1 − S = 0.001 if every side that is alive at initialization were learned. Is initialization the
only obstacle?

<details><summary>Solution</summary>

About half of the 200 sides are alive at initialization, yet far fewer are computed: being
alive is nearly necessary (dead sides are almost never learned) but not sufficient, because
the alive sides also compete for 40 neurons. The bias-init variant, where no side starts dead,
improves the loss but still does not reach m/2 features computed well: initialization explains
part of the gap, not all of it.
</details>